# Agent Repair Pipeline — HotpotQA

**ICLR Experiment Pipeline — Colab A100-80GB + Qwen2.5-32B-Instruct-AWQ**

| Stage | What it does | ~Time (A100-80GB, 500 Qs) |
|---|---|---|
| 0 | Download HotpotQA, create folders | 1 min |
| 1 | ReAct trajectory generation (Qwen2.5-32B-Instruct-AWQ) | ~1-2 h |
| 2 | Step-level uncertainty scoring | ~1-2 h |
| 3 | 72B judge error annotation | ~1-2 h |
| 4 | Localization scoring (+ cascade-aware rules) | < 1 min |
| 5 | 126 repair strategies (backtrack × nudge type ablation) | ~4-6 h |
| 6 | Tables, stats, figures, ablation analysis | < 1 min |

**Every stage is resumable** — if Colab disconnects, re-run the cell and it picks up where it left off.

---
## 0. Setup

In [1]:
# Verify GPU — should show A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_hotpotqa.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/agent-repair'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 203 (delta 104), reused 164 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 261.91 KiB | 1.26 MiB/s, done.
Resolving deltas: 100% (104/104), done.
Working directory: /content/agent-repair


In [4]:
# Check Colab's CUDA version first
!nvcc --version 2>/dev/null || echo "nvcc not found"
!python -c "import torch; print(f'torch CUDA: {torch.version.cuda}')" 2>/dev/null || echo "torch not installed yet"
!nvidia-smi | grep "CUDA Version"

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
torch CUDA: 12.8
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |


In [5]:
# Install vLLM — pinned to 0.19.0 (last version with CUDA 12 default wheels)
# vLLM >= 0.20 ships CUDA 13 wheels which don't work on Colab's CUDA 12.x
!pip uninstall -y vllm 2>/dev/null
!pip install -q "vllm==0.19.0" 2>&1 | tail -5

# Install remaining dependencies
!pip install -q "transformers>=4.57.0" "tokenizers>=0.21.0" accelerate \
    huggingface_hub datasets scipy scikit-learn pandas pyarrow \
    pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

print('\n--- Installation complete ---')

gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.

--- Installation complete ---


In [6]:
# Quick sanity check: can vLLM see the GPU?
import torch, vllm
print(f'torch:  {torch.__version__}')
print(f'vLLM:   {vllm.__version__}')
print(f'CUDA:   {torch.version.cuda}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch:  2.10.0+cu128
vLLM:   0.19.0
CUDA:   12.8
GPU:    NVIDIA A100-SXM4-80GB
VRAM:   85.1 GB


In [7]:
# Clear previous data
import shutil, os
drive_base = '/content/drive/MyDrive/agent-repair-hotpotqa'
for d in ['outputs', 'data/processed']:
    p = os.path.join(drive_base, d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
for d in ['outputs', 'data/processed']:
    p = os.path.join('/content/agent-repair', d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
print('Ready for fresh run with HotpotQA')

Ready for fresh run with HotpotQA


---
## Smoke Test (20 questions, ~10-15 min)

**Run this first** to verify everything works before committing A100 units to the full 500-question run.

In [8]:
stages = ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
          'run_localize', 'run_repair', 'run_eval']

for stage in stages:
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG} --limit 20
    print()


  run_setup
05:42:10 | INFO    | setup | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
05:42:10 | INFO    | setup | Dataset: hotpotqa (file: hotpot_dev_distractor_v1.json)
Trying direct: http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json
  failed: <urlopen error timed out>
Trying direct: https://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json
  failed: <urlopen error timed out>
05:42:50 | INFO    | setup | Direct download unavailable -> Hugging Face fallback
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpotqa/hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 9.52kB [00:00, 38.5MB/s]
distractor/train-00000-of-00002.parquet: 100% 166M/166M [00:02<00:00, 59.2MB/s]
distractor/train-00001-of-

In [9]:
# Check smoke test produced output
import glob
tables = glob.glob('outputs/tables/*') + glob.glob('/content/drive/MyDrive/agent-repair-hotpotqa/outputs/tables/*')
figs = glob.glob('outputs/figures/*') + glob.glob('/content/drive/MyDrive/agent-repair-hotpotqa/outputs/figures/*')
print(f'Tables: {len(tables)}, Figures: {len(figs)}')
if tables or figs:
    print('Smoke test PASSED — safe to run the full pipeline below.')
else:
    print('WARNING: No outputs found. Check the logs above for errors.')

Tables: 4, Figures: 0
Smoke test PASSED — safe to run the full pipeline below.


---
## Full Pipeline (500 questions)

Only run these cells after the smoke test passes. To reset and run fresh, delete the `data/processed/` folder.

### Stage 0: Download HotpotQA

In [10]:
!python scripts/run_setup.py --config {CONFIG}

05:59:00 | INFO    | setup | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
05:59:00 | INFO    | setup | Dataset: hotpotqa (file: hotpot_dev_distractor_v1.json)
05:59:00 | INFO    | setup | Dataset already present: /content/drive/MyDrive/agent-repair-hotpotqa/data/raw/hotpot_dev_distractor_v1.json
05:59:00 | INFO    | setup | Dataset ready: /content/drive/MyDrive/agent-repair-hotpotqa/data/raw/hotpot_dev_distractor_v1.json (46.8 MB, 7405 questions)
05:59:00 | INFO    | setup | Output dirs created under: /content/drive/MyDrive/agent-repair-hotpotqa


### Stage 1: Generate ReAct Trajectories

In [11]:
!python scripts/run_generate.py --config {CONFIG}

05:59:01 | INFO    | stage1 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
05:59:01 | INFO    | stage1 | Dataset: hotpotqa (env: HotpotEnv)
05:59:01 | INFO    | stage1 | Pool: 500 questions
05:59:01 | INFO    | stage1 | To process: 480 of 500 (20 already done)
05:59:03 | INFO    | stage1 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-21 05:59:06.481553: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 05:59:06.553007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebui

### Stage 2: Step-Level Uncertainty

In [12]:
!python scripts/run_uncertainty.py --config {CONFIG}

06:05:21 | INFO    | stage2 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
06:05:21 | INFO    | stage2 | Math metrics: 480 of 500 trajectories to do
06:05:34 | INFO    | stage2 | Math metrics done.
06:05:34 | INFO    | stage2 | Sampling metrics: 217 of 226 failed trajectories
06:05:35 | INFO    | stage2 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-21 06:05:38.814231: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 06:05:38.884507: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other o

### Stage 3: Error Annotation (72B Judge)

In [13]:
!python scripts/run_annotate.py --config {CONFIG}

06:13:48 | INFO    | stage3 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
06:13:48 | INFO    | stage3 | To annotate: 217 of 226 failed trajectories
06:13:49 | INFO    | stage3 | GPU VRAM: 85.094825984 GB | judge: Qwen/Qwen2.5-72B-Instruct-AWQ (85 GB -> 72B judge)
2026-07-21 06:13:52.921100: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 06:13:52.991437: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 07-21 06:14:03 [utils.py:233] non-default args: {'trust_r

### Stage 4: Localization Scoring

In [14]:
!python scripts/run_localize.py --config {CONFIG}

06:20:07 | INFO    | stage4 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
06:20:08 | INFO    | stage4 | records: 2260 (226 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.248           0.460  0.456          0.279              0.553     0.389     0.584  226
token_entropy_max             0.195           0.451  0.420          0.230              0.544     0.376     0.544  226
max_token_prob_max            0.177           0.442  0.418          0.248              0.549     0.389     0.566  226
perplexity                    0.173           0.456  0.406          0.212              0.562     0.332     0.562  226
max_token_prob_mean           0.164           0.447  0.400          0.208              0.562  

### Stage 5: Repair (126 Strategies)

In [15]:
!python scripts/run_repair.py --config {CONFIG}

06:20:09 | INFO    | stage5 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
06:20:09 | INFO    | stage5 | Dataset: hotpotqa (env: HotpotEnv)
06:20:09 | INFO    | stage5 | 226 failed x 126 strategies x 3 seeds = 85428 result rows (actual GPU runs are far fewer: dedup)
06:20:11 | INFO    | stage5 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-21 06:20:14.469302: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 06:20:14.542563: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations,

### Stage 6: Evaluation

In [16]:
!python scripts/run_eval.py --config {CONFIG}

07:42:30 | INFO    | stage6 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
07:42:32 | INFO    | stage6 | rows: 86106 | uncertainty strategies: 120 (bt0: 60) | + ensemble

MAIN RESULTS  (fixed_% = share of FAILED trajectories that the repair fixed)
                                                           strategy  fixed_%     95%_CI  avg_tokens  avg_tool_calls  hit_true_step_%
                                                        random_step      8.4  6.3-10.5%         180            2.78             23.0
                                                       full_restart     14.6 11.9-17.3%         275            4.75             31.9
                                               oracle_targeted__bt2     12.2  9.9-14.8%         272            4.51             31.9
                                     oracle_targeted__bt2__informed     11.8  9.4-14.3%         276            4.58             31.9
                                        

---
## Results

In [17]:
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

Summary not yet generated. Run Stage 6 first.


In [18]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,8.4,6.3-10.5%,180,2.78,23.0
1,full_restart,14.6,11.9-17.3%,275,4.75,31.9
2,oracle_targeted__bt2,12.2,9.9-14.8%,272,4.51,31.9
3,oracle_targeted__bt2__informed,11.8,9.4-14.3%,276,4.58,31.9
4,oracle_targeted__informed,11.4,9.0-13.7%,244,3.83,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,12.1,9.7-14.6%,263,4.36,23.0
123,unc__verbalized_confidence__topk__bt2__informed,12.7,10.2-15.2%,264,4.38,23.0
124,unc__verbalized_confidence__topk__informed,10.8,8.4-13.1%,236,3.92,18.6
125,uncertainty_ensemble_any,36.4,32.9-40.1%,12836,205.14,89.4


In [19]:
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')

No figures yet. Run Stage 6 first.


---
## Re-run Stages 4–6 with Cascade-Aware Strategies

**Use this after you already have stages 1–3 completed** (trajectories, uncertainty, annotations).
This pulls the latest code (with cascade rules), clears only localization/repair/eval outputs,
and re-runs stages 4→5→6 in one cell. ~3-4 hours on A100-80GB for 500 questions.

In [20]:
import os, shutil

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_hotpotqa.yaml'

# 1. Pull latest code with cascade-aware strategies
os.chdir(REPO_DIR)
!git pull

# 2. Clear ONLY stages 4-6 outputs (keep trajectories, uncertainty, annotations)
drive_base = '/content/drive/MyDrive/agent-repair-hotpotqa'
for base in [REPO_DIR, drive_base]:
    for d in ['outputs/localization', 'outputs/repairs', 'outputs/tables', 'outputs/figures']:
        p = os.path.join(base, d)
        if os.path.exists(p):
            shutil.rmtree(p)
            print(f'Cleared {p}')

# 3. Clear checkpoint files for stages 4-6 (these live in outputs/logs/)
for base in [REPO_DIR, drive_base]:
    for ckpt in ['outputs/logs/stage4_localize.jsonl',
                 'outputs/logs/stage5_repair.jsonl',
                 'outputs/logs/stage6_eval.jsonl']:
        p = os.path.join(base, ckpt)
        if os.path.exists(p):
            os.remove(p)
            print(f'Removed checkpoint: {p}')

print('\nStages 1-3 data preserved. Ready to re-run 4→5→6.')

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.46 KiB | 1.46 MiB/s, done.
From https://github.com/kishormorol/agent-repair
   a76bf73..aba0a7c  main       -> origin/main
Updating a76bf73..aba0a7c
Fast-forward
 paper_title_abstract.md | 9 +++++++++
 1 file changed, 9 insertions(+)
 create mode 100644 paper_title_abstract.md
Cleared /content/drive/MyDrive/agent-repair-hotpotqa/outputs/localization
Cleared /content/drive/MyDrive/agent-repair-hotpotqa/outputs/repairs
Cleared /content/drive/MyDrive/agent-repair-hotpotqa/outputs/tables
Cleared /content/drive/MyDrive/agent-repair-hotpotqa/outputs/figures
Removed checkpoint: /content/drive/MyDrive/agent-repair-hotpotqa/outputs/logs/stage5_repair.jsonl

Stages 1-3 data preserved. Ready to re-run 4→5→6.


In [21]:
# Run stages 4 → 5 → 6 sequentially (run this and go to sleep)
import time

for stage in ['run_localize', 'run_repair', 'run_eval']:
    t0 = time.time()
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG}
    elapsed = (time.time() - t0) / 60
    print(f'  ✓ {stage} done in {elapsed:.1f} min\n')

print('\n' + '='*60)
print('  ALL DONE — check results below')
print('='*60)


  run_localize
07:42:43 | INFO    | stage4 | config=config/config_colab_hotpotqa.yaml  base=/content/drive/MyDrive/agent-repair-hotpotqa
07:42:44 | INFO    | stage4 | records: 2260 (226 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.248           0.460  0.456          0.279              0.553     0.389     0.584  226
token_entropy_max             0.195           0.451  0.420          0.230              0.544     0.376     0.544  226
max_token_prob_max            0.177           0.442  0.418          0.248              0.549     0.389     0.566  226
perplexity                    0.173           0.456  0.406          0.212              0.562     0.332     0.562  226
max_token_prob_mean           0.164           0.447  0.400          0.208     

In [22]:
# View cascade-aware results
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())

import pandas as pd
results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)

import glob
from IPython.display import display, Image
figs = sorted(glob.glob(os.path.join(cfg.path('figures'), '*.png')))
for f in figs:
    print(f'\n--- {os.path.basename(f)} ---')
    display(Image(filename=f, width=800))

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,8.1,6.0-10.2%,182,2.80,23.0
1,full_restart,14.2,11.7-16.8%,274,4.74,31.9
2,oracle_targeted__bt2,12.4,10.0-14.9%,272,4.51,31.9
3,oracle_targeted__bt2__informed,11.8,9.4-14.2%,276,4.60,31.9
4,oracle_targeted__informed,11.4,9.0-13.9%,244,3.85,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,12.1,9.7-14.6%,262,4.36,23.0
123,unc__verbalized_confidence__topk__bt2__informed,12.7,10.3-15.2%,265,4.40,23.0
124,unc__verbalized_confidence__topk__informed,10.3,8.1-12.7%,236,3.93,18.6
125,uncertainty_ensemble_any,36.0,32.5-39.7%,12868,206.16,89.4
